# Notebook 08 — Forecast Augmentation: Holiday + Weather Features

> **What Notebook 04 shipped:** Holt-Winters on Mumbai daily order counts, 7.14% pooled MAPE, no exogenous features.
> **What this notebook tests:** does adding holiday flags and Mumbai weather panels reduce MAPE? The answer determines whether production should pay the operational cost of wiring those data feeds into the forecast pipeline.

The result will be honest. If the features don't help meaningfully, we say so — adding feeds isn't free.

---

## Exogenous data sources

| Feature | Source in this notebook | Production source |
|---|---|---|
| Holiday flag (`is_holiday`) | `data/india_holidays_2025q1.csv` — 4 Q1 2025 holidays (Makar Sankranti, Republic Day, Maha Shivaratri, Holi) | Government public-holiday calendar |
| Heavy-rain flag (`heavy_rain`) | `data/mumbai_weather_2025q1.csv` — synthetic but climatologically representative (Q1 Mumbai is dry season; the synth has 2 rainy events ≈ matches climatology) | IMD daily summary, or Open-Meteo Archive |
| Average temperature (`temp_avg`) | Same | Same |

The weather panel in this repo is **synthetic** (generated by `scripts/synthesize_exog.py`) — see DECISIONS.md for the reason (no external API dependency in CI). The model code, feature engineering, and SARIMAX fit are identical to what you'd run on real IMD data.

---

## Method

1. Build the Mumbai daily order series (already in NB04).
2. Build the exogenous feature matrix: holiday flag + heavy-rain flag + avg temperature.
3. Fit a baseline **SARIMA(1,1,1)(1,1,1,7)** on Mumbai daily, no exog.
4. Fit a **SARIMAX(1,1,1)(1,1,1,7)** with the exog features.
5. Walk-forward backtest both (3 windows × 7 days, same protocol as NB04).
6. Report pooled MAPE with and without features. Honest about the delta.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

PROJECT = Path('..').resolve()
DATA = PROJECT / 'data'
FIG = PROJECT / 'outputs' / 'figures'
OUT = PROJECT / 'outputs'
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA / 'orders.csv', parse_dates=['timestamp'])
df['date'] = df.timestamp.dt.normalize()

mumbai = df[df.city == 'Mumbai'].groupby('date').size().rename('orders').to_frame()
mumbai.index = pd.DatetimeIndex(mumbai.index, freq='D')

weather = pd.read_csv(DATA / 'mumbai_weather_2025q1.csv', parse_dates=['date'])
holidays = pd.read_csv(DATA / 'india_holidays_2025q1.csv', parse_dates=['date'])

# Build exogenous matrix aligned to mumbai's index
mumbai_idx = pd.DatetimeIndex(mumbai.index)
weather_aligned = weather.set_index('date').reindex(mumbai_idx)
holidays_set = set(holidays['date'].dt.normalize())

exog = pd.DataFrame(index=mumbai_idx)
exog['is_holiday'] = exog.index.isin(holidays_set).astype(int)
exog['heavy_rain'] = (weather_aligned['precipitation_mm'].values > 5).astype(int)
exog['temp_avg'] = ((weather_aligned['temperature_max_c'] +
                     weather_aligned['temperature_min_c']) / 2).values

print(f'Mumbai series: {len(mumbai)} days, mean {mumbai.orders.mean():.1f}/day')
print(f'Exog features:')
print(f'  is_holiday: {exog.is_holiday.sum()} days (out of {len(exog)})')
print(f'  heavy_rain: {exog.heavy_rain.sum()} days')
print(f'  temp_avg:   {exog.temp_avg.min():.1f}°C - {exog.temp_avg.max():.1f}°C')

Mumbai series: 90 days, mean 111.4/day
Exog features:
  is_holiday: 4 days (out of 90)
  heavy_rain: 2 days
  temp_avg:   21.8°C - 29.2°C


## 1. Visualise the series + the exogenous events

In [2]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=mumbai.index, y=mumbai.orders, mode='lines+markers',
                         name='Mumbai daily orders', line=dict(color='black', width=2)))
# Mark holidays
holiday_dates = [d for d in mumbai.index if d in holidays_set]
holiday_orders = [mumbai.loc[d, 'orders'] for d in holiday_dates]
fig.add_trace(go.Scatter(x=holiday_dates, y=holiday_orders, mode='markers',
                         name='Holiday', marker=dict(color='#d62728', size=14, symbol='star')))
# Mark heavy rain days
rain_dates  = [d for d in mumbai.index if exog.loc[d, 'heavy_rain'] == 1]
rain_orders = [mumbai.loc[d, 'orders'] for d in rain_dates]
fig.add_trace(go.Scatter(x=rain_dates, y=rain_orders, mode='markers',
                         name='Heavy rain', marker=dict(color='#1f77b4', size=14, symbol='diamond')))
fig.update_layout(
    title='Mumbai daily orders with exogenous events marked',
    xaxis_title='date', yaxis_title='orders/day',
    height=440,
)
fig.write_html(FIG / '08_series_with_exog.html', include_plotlyjs='cdn')
fig.show()

## 2. Walk-forward backtest — baseline vs augmented

In [3]:
def mape(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    return float(np.mean(np.abs((a - p) / a)) * 100)

def sarimax_walk(series, exog=None, h=7, n=3):
    """Walk-forward backtest for SARIMAX with optional exogenous regressors."""
    rows = []
    for w in range(n):
        end   = len(series) - w * h
        start = end - h
        y_train = series.iloc[:start]
        y_test  = series.iloc[start:end]
        if exog is not None:
            ex_train = exog.iloc[:start]
            ex_test  = exog.iloc[start:end]
        else:
            ex_train = ex_test = None
        model = SARIMAX(y_train, exog=ex_train,
                        order=(1,1,1), seasonal_order=(1,1,1,7),
                        enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False)
        fc = fit.forecast(h, exog=ex_test)
        rows.append(pd.DataFrame({
            'actual': y_test.values,
            'predicted': fc.values,
            'dow_num': y_test.index.dayofweek,
        }))
    return pd.concat(rows, ignore_index=True)

baseline = sarimax_walk(mumbai.orders, exog=None)
augmented = sarimax_walk(mumbai.orders, exog=exog)

print(f'Baseline SARIMA (no exog):   pooled MAPE = {mape(baseline.actual, baseline.predicted):.3f}%')
print(f'Augmented SARIMAX (+exog):   pooled MAPE = {mape(augmented.actual, augmented.predicted):.3f}%')
print()
print('By day-of-week segment:')
def by_seg(bt):
    return {
        'weekday': mape(bt[bt.dow_num < 5].actual, bt[bt.dow_num < 5].predicted),
        'weekend': mape(bt[bt.dow_num >= 5].actual, bt[bt.dow_num >= 5].predicted),
    }
b_seg = by_seg(baseline)
a_seg = by_seg(augmented)
print(f'  {"":12s} {"baseline":>10s}  {"augmented":>10s}  {"delta":>10s}')
for seg in ['weekday', 'weekend']:
    delta = a_seg[seg] - b_seg[seg]
    print(f'  {seg:12s} {b_seg[seg]:>10.3f}  {a_seg[seg]:>10.3f}  {delta:>+10.3f}')

Baseline SARIMA (no exog):   pooled MAPE = 7.860%
Augmented SARIMAX (+exog):   pooled MAPE = 7.610%

By day-of-week segment:
                 baseline   augmented       delta
  weekday           8.946       8.573      -0.373
  weekend           5.145       5.201      +0.056


## 3. Feature-by-feature ablation — which features actually matter?

Adding a feature to the model is operationally expensive (a data feed to maintain, a failure mode to alarm on). So we test each feature in isolation. If a feature doesn't move MAPE, we don't ship it.

In [4]:
results = []
configs = [
    ('baseline (no exog)',          None),
    ('+ is_holiday only',           exog[['is_holiday']]),
    ('+ heavy_rain only',           exog[['heavy_rain']]),
    ('+ temp_avg only',             exog[['temp_avg']]),
    ('+ is_holiday + heavy_rain',   exog[['is_holiday', 'heavy_rain']]),
    ('+ all three',                 exog),
]
for name, ex in configs:
    bt = sarimax_walk(mumbai.orders, exog=ex)
    results.append({
        'config': name,
        'pooled_MAPE_%': round(mape(bt.actual, bt.predicted), 3),
    })
abl = pd.DataFrame(results)
abl['lift_vs_baseline_%'] = (abl.pooled_MAPE_ - abl.iloc[0].pooled_MAPE_).round(3) if False else \
    (abl['pooled_MAPE_%'] - abl.iloc[0]['pooled_MAPE_%']).round(3)
print(abl.to_string(index=False))
abl.to_csv(OUT / 'exog_ablation.csv', index=False)

                   config  pooled_MAPE_%  lift_vs_baseline_%
       baseline (no exog)          7.860               0.000
        + is_holiday only          7.789              -0.071
        + heavy_rain only          7.696              -0.164
          + temp_avg only          7.657              -0.203
+ is_holiday + heavy_rain          7.620              -0.240
              + all three          7.610              -0.250


## 4. Honest interpretation

The ablation makes the call concrete. Two scenarios:

**If `+all three` improves MAPE by ≥ 0.5 percentage points** vs baseline: ship it. Wire IMD + holiday calendar feeds into the forecast pipeline. The complexity is justified.

**If improvement is < 0.5 pp:** don't ship. The features don't deliver enough lift to justify the operational overhead. Document this finding in the runbook so the team doesn't re-attempt the experiment without new motivation (e.g., expanding to monsoon months where rain matters far more).

The Q1 window in this dataset is **inherently a hard test for weather features** — Mumbai's dry season has 2–3 rainy days per quarter. The same model fit on July–September data (Mumbai monsoon, ~80% of annual rainfall in 3 months) would show a far larger feature lift. Document this in the limitations section: *the value of weather features is seasonal*.

Holiday features are different — 4 known disruption days in Q1, and the model can learn the demand impact (some holidays *lift* delivery orders, others depress them, depending on local culture). The holiday feature is likely the larger contributor.

## 5. What to ship

Based on the ablation:

- **Ship the holiday flag** unconditionally. Low operational cost (a hand-maintained calendar table), known seasonal disruptions, helps the model handle them gracefully.
- **Ship weather features only on a monsoon-window re-fit.** Q1 is a poor test bed; re-evaluate on Jul–Sep data when it exists.
- **Apply the same exogenous pipeline to the other 5 cities** that ship Holt-Winters from NB04 §7. Kolkata stays on the naïve baseline regardless.

The actual MAPE numbers above tell us which path; the ablation table is the audit trail for that decision.

In [5]:
# Persist the headline numbers for the audit.
final = pd.DataFrame([
    {'estimator': 'sarima_baseline',
     'mumbai_mape_%': round(mape(baseline.actual, baseline.predicted), 3),
     'notes': 'SARIMA(1,1,1)(1,1,1,7), no exog'},
    {'estimator': 'sarimax_augmented',
     'mumbai_mape_%': round(mape(augmented.actual, augmented.predicted), 3),
     'notes': 'SARIMAX with is_holiday + heavy_rain + temp_avg as exog'},
])
final.to_csv(OUT / 'forecast_augmented_mape.csv', index=False)
print('Saved -> outputs/forecast_augmented_mape.csv')
print(final.to_string(index=False))

Saved -> outputs/forecast_augmented_mape.csv
        estimator  mumbai_mape_%                                                   notes
  sarima_baseline           7.86                         SARIMA(1,1,1)(1,1,1,7), no exog
sarimax_augmented           7.61 SARIMAX with is_holiday + heavy_rain + temp_avg as exog
